# Prepare MIMIC-III Data

Preparing the MIMIC-III data consists of adding restricted information from the MIMIC-III clinical database to `data/mimic3_samples_pre.h5`.

In [ ]:
import lab_ecg
import numpy as np
import pandas as pd
import tqdm

In [ ]:
samples = pd.read_hdf("data/mimic3_samples_pre.h5", key="table")

### Step 0: Download MIMIC-III

Access to the MIMIC-III clinical database is restricted. After being granted access, you can download it from [PhysioNet](https://physionet.org/content/mimiciii/1.4/).

In [ ]:
mimic3_root = (
    ...
)  # set to root directory of your local copy of MIMIC-III; must contain "PATIENTS.csv" and "LABEVENTS.csv"

### Step 1: Add Patient Demographics: Age & Gender

In [ ]:
patients = pd.read_csv(mimic3_root + "/PATIENTS.csv", parse_dates=["DOB"])

In [ ]:
demo = patients.set_index("SUBJECT_ID")[["DOB", "GENDER"]].reindex(samples["SUBJECT_ID"])
demo.index = samples.index

In [ ]:
samples["male"] = demo["GENDER"] == "M"

In [ ]:
samples["male"].mean()

In [ ]:
samples["age"] = (samples["timestamp"] - pd.Timestamp(0)) / pd.Timedelta(365.2525, unit="d") - (
    demo["DOB"] - pd.Timestamp(0)
) / pd.Timedelta(365.2525, unit="d")

# account for de-identified DoBs
samples.loc[samples["age"] > 200, "age"] = 91.4 + samples.loc[samples["age"] > 200, "age"] - 300

In [ ]:
samples["age"].describe()

### Step 2: Add Laboratory Measurements

In [ ]:
itemids = [
    i
    for k, v in lab_ecg.MIMIC_LAB_MAPPING.items()
    if k
    in (
        "albumin",
        "BUN",
        "total_calcium",
        "bilirubin",
        "glucose",
        "hematocrit",
        "hemoglobin",
        "INR",
        "potassium",
        "creatinine",
        "phosphate",
        "leukocytes",
        "pO2",
    )
    for i in v
]
len(itemids)

In [ ]:
# LABEVENTS.csv is huge => load iteratively
iter_csv = pd.read_csv(
    mimic3_root + "/LABEVENTS.csv",
    parse_dates=["charttime"],
    iterator=True,
    chunksize=1000000,
    dtype={"valuenum": np.float32},
)
lab = pd.concat(
    [
        chunk.loc[
            chunk["subject_id"].isin(samples["SUBJECT_ID"].unique()) & chunk["itemid"].isin(itemids),
            ["subject_id", "itemid", "charttime", "valuenum"],
        ]
        for chunk in tqdm.tqdm(iter_csv)
    ],
    axis=0,
)

In [ ]:
# take median of concurrent values
lab = lab.groupby(["subject_id", "charttime", "itemid"])["valuenum"].median().to_frame("valuenum")
lab.reset_index(inplace=True)

In [ ]:
lab = lab_ecg.validate_lab(lab)

In [ ]:
samples = samples.join(
    lab_ecg.assign_labs_to_records(lab, samples, temporal_dist_minutes=30, records_id_col="SUBJECT_ID"),
    how="left",
)

### Step 3: Save

In [ ]:
samples.to_hdf("data/mimic3_samples.h5", key="table", complevel=9, mode="w")

### Step 4: Descriptive Statistics: Parameter/Threshold Counts & Base Rates

In [ ]:
lst = []
for param, threshold, pretty in lab_ecg.TARGETS:
    if param in samples.columns:
        mask = (samples[param + "_split"] == "test") & samples[param].notna()
        if threshold.startswith("<"):
            br = (samples.loc[mask, param] < float(threshold[1:])).mean()
        else:
            br = (samples.loc[mask, param] > float(threshold[1:])).mean()
        lst.append((pretty, f"{mask.sum():d} ({br * 100:.2f}%)"))

In [ ]:
pd.DataFrame(lst).set_index(0)[1].sort_index()